[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/biology_aging/blob/main/notebooks/build-telomere-weak-biomarker.ipynb)

# A Real Number That Can't Tell Your Age 📏
### A hands-on companion to Chapter 1

Telomeres — the caps on your chromosomes — genuinely shorten as you age. So can we read your age, or your "biological age," straight off your telomere length? In this notebook you'll simulate a realistic population and discover that the honest answer is mostly **no**: the scatter is so wide that telomere length barely pins down an individual. Then you'll see what a *strong* predictor (the Chapter 2 clock) looks like by comparison.

The lesson is one of the most important in the whole book: **a number can be real at the population level and still be useless for you.**

No setup. Runs in Colab. Press ▶ top to bottom.

## Step 0 — Tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
print("Ready.")

## Part 1 — Telomeres really do shorten with age

We'll simulate **800 people**. Each is born with a telomere length drawn from a wide range (you *inherit* your starting length, and people differ a lot), then loses roughly **45 base pairs per year**, plus biological noise.

In [ ]:
n = 800
age = rng.uniform(20, 80, n)
start_length = rng.normal(9.8, 1.6, n)               # inherited start (kb) - big spread
telomere = start_length - 0.045*age + rng.normal(0, 0.4, n)   # ~45 bp/yr loss + noise

r = np.corrcoef(telomere, age)[0,1]
print(f"Correlation between telomere length and age: r = {r:.2f}  (real-world: about -0.4)")

slope, intercept = np.polyfit(age, telomere, 1)
plt.figure(figsize=(7,4.5))
plt.scatter(age, telomere, s=10, alpha=0.4, color="#457b9d")
plt.plot(age, slope*age+intercept, color="#1d3557", lw=2, label="average trend")
plt.xlabel("age (years)"); plt.ylabel("telomere length (kb)")
plt.title("Yes - on average, telomeres shorten with age"); plt.legend(); plt.tight_layout(); plt.show()

The downward trend is real. A naive headline would stop here: *"telomeres shorten with age, so measure yours to know your biological age!"* Let's test whether that actually works.

## Part 2 — But can it tell *your* age?

Flip the question around: given someone's telomere length, predict their age. We fit that line and measure two things — the typical error, and the **prediction band** (the range of ages consistent with a given telomere length).

In [ ]:
# predict age from telomere length
b, a0 = np.polyfit(telomere, age, 1)
pred_age = b*telomere + a0
mae   = np.mean(np.abs(age - pred_age))
band  = 1.96*np.std(age - pred_age)          # ~95% prediction band
print(f"Predicting age from telomere length: typical error = {mae:.0f} years")
print(f"R^2 = {r**2:.2f}  (telomere length explains only ~{r**2*100:.0f}% of the age differences)")

# what does one person's telomere length tell us about their age?
example = np.median(telomere)
center  = b*example + a0
print(f"\nSomeone with a telomere length of {example:.1f} kb could plausibly be anywhere")
print(f"from age {center-band:.0f} to {center+band:.0f} - a {2*band:.0f}-year window.")

order = np.argsort(telomere)
plt.figure(figsize=(7,4.5))
plt.scatter(telomere, age, s=10, alpha=0.35, color="#457b9d")
plt.plot(telomere[order], (b*telomere+a0)[order], color="#1d3557", lw=2, label="best guess")
plt.fill_between(telomere[order], (b*telomere+a0-band)[order], (b*telomere+a0+band)[order],
                 color="#a8dadc", alpha=0.5, label="ages that fit")
plt.xlabel("telomere length (kb)"); plt.ylabel("true age (years)")
plt.title(f"A real signal, drowned in scatter (off by ~{mae:.0f} years)"); plt.legend()
plt.tight_layout(); plt.show()

Knowing your telomere length barely narrows down your age — the band of "ages that fit" is enormous. The signal is *real* but *weak*: real enough to see across a whole population, far too noisy to tell you much about **you**.

## Part 3 — What a *strong* predictor looks like

For contrast, here's a strong biomarker — standing in for the **epigenetic clock** of Chapter 2 — measured on the same people. Same plot, same band. Watch the difference.

In [ ]:
clock = age + rng.normal(0, 3.0, n)                 # a tight, strong predictor of age
rc = np.corrcoef(clock, age)[0,1]
bc, ac = np.polyfit(clock, age, 1)
pred_c = bc*clock + ac
mae_c  = np.mean(np.abs(age - pred_c))
band_c = 1.96*np.std(age - pred_c)

print(f"Telomeres      : error ~{mae:.0f} yr,  R^2 = {r**2:.2f}")
print(f"Clock (Ch.2)   : error ~{mae_c:.0f} yr,  R^2 = {rc**2:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11,4.3), sharey=True)
for a_, (x, line_b, line_a, bd, ttl, col) in zip(
        ax, [(telomere, b, a0, band, f"Telomeres (off by ~{mae:.0f} yr)", "#457b9d"),
             (clock,    bc, ac, band_c, f"Epigenetic clock (off by ~{mae_c:.0f} yr)", "#2a9d8f")]):
    o = np.argsort(x)
    a_.scatter(x, age, s=8, alpha=0.3, color=col)
    a_.plot(x[o], (line_b*x+line_a)[o], color="#1d3557", lw=2)
    a_.fill_between(x[o], (line_b*x+line_a-bd)[o], (line_b*x+line_a+bd)[o], color=col, alpha=0.25)
    a_.set_title(ttl); a_.set_xlabel("biomarker value")
ax[0].set_ylabel("true age (years)")
plt.tight_layout(); plt.show()

Same idea, opposite usefulness. Both telomeres and the clock are "biomarkers of aging," but one is a thermometer that barely registers and the other reads your age to within a few years. That gap is *exactly* why the field needed the epigenetic clock, and why *"measure your telomeres to learn your biological age"* oversells a genuinely weak signal.

> Keep two lessons side by side. **Real on average ≠ informative for you** (this notebook). And even the *strong* clock is still only a **predictor, not a lever** — moving it isn't the same as changing your aging, as you saw in the Chapter 2 notebook. A good biomarker has to clear *both* bars.

### Go further
- Widen or narrow the inherited starting-length spread (`1.6`). How tight would the scatter have to be before telomeres became useful for individuals?
- How many people would you need to *reliably* detect a 2-year telomere difference between two groups? (Real telomere studies need thousands.)
- Combine telomere length with a few other weak markers and see whether, together, they start to predict age better.